In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API Key is set.")

API Key is set.


In [2]:
from langchain_groq import ChatGroq

c:\Desktop\Dhanush\RAG_Application\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

## **RAG Implementation with your own text data**

#### **Step 1: Preparing Document for your text**

In [4]:
from langchain_core.documents import Document

##Your text
my_text = """LangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.[3][4]

In the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.[5][6]

In October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API.[7]

In February 2024 LangChain released LangSmith, a closed-source observability and evaluation platform for LLM applications, and announced a US $25 million Series A led by Sequoia Capital.[8] On 14 May 2025 the company launched LangGraph Platform into general availability, providing managed infrastructure for deploying long-running, stateful AI agents.[9]

In April 2025, LangChain was featured in the Forbes AI 50 list."""

docs = [Document(page_content=my_text, metadata={"source": "ABC", "documentID": "Doc1"})]
docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='LangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.[3][4]\n\nIn the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.[5][6]\n\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API.[7]\n\nIn February 2024 LangChain released LangSmith, a closed-source observability and evaluation platform for LLM applications, and announced a US $25 million Series A led by Sequoia Capital.[8] On 14 May 2025 the company launched LangGraph Platform into gen

#### **Step 2: Splitting the document into chunks**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='LangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.[3][4]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='In the third quarter of 2023, the LangChain Expression Language (LCEL) was introduced, which provides a declarative way to define chains of actions.[5][6]\n\nIn October 2023 LangChain introduced LangServe, a deployment tool to host LCEL code as a production-ready API.[7]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='In February 2024 LangChain released LangSmith, a closed-source observability and evaluation platform for 

#### **Step 3: Creating Embeddings for the chunks**

In [6]:
# Using HuggingFace embeddings — completely free!
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2522.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embedding_model.embed_query("what is langchain?") 

[-0.04055778682231903,
 0.013319544494152069,
 0.019409174099564552,
 0.0048216828145086765,
 -0.06119084358215332,
 0.015379196964204311,
 0.054146233946084976,
 0.05520950257778168,
 0.07971914112567902,
 -0.024081295356154442,
 0.08297760784626007,
 -0.016691358759999275,
 -0.029097484424710274,
 0.02071719616651535,
 -0.06865743547677994,
 -0.020465929061174393,
 -0.07295339554548264,
 0.05014486238360405,
 0.010502133518457413,
 -0.07019737362861633,
 -0.03522021323442459,
 0.01595531776547432,
 0.0465620793402195,
 0.03276348114013672,
 0.049144335091114044,
 -0.07498356699943542,
 0.014402812346816063,
 -0.014078278094530106,
 0.051738448441028595,
 0.003820236772298813,
 0.009939463809132576,
 0.0718270093202591,
 -0.012822604738175869,
 0.014195708557963371,
 -0.1152329221367836,
 0.12732365727424622,
 -0.008469412103295326,
 -0.007185508497059345,
 0.03286218270659447,
 0.017457948997616768,
 -0.053777240216732025,
 -0.00045475777005776763,
 0.05582141503691673,
 -0.052348252

#### **Step 4: Create and store EMbeddings in Vector Store**

In [8]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

#### **Step 5: Semantic Search**

In [9]:
vectorstore.similarity_search("what is langchain?", k=3)

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='LangChain was launched in October 2022 as an open source project by Harrison Chase, while working at machine learning startup Robust Intelligence. In April 2023, LangChain had incorporated and the new startup raised over $20 million in funding at a valuation of at least $200 million from venture firm Sequoia Capital, a week after announcing a $10 million seed investment from Benchmark.[3][4]')]

#### **Talk to LLM**

In [10]:
context = vectorstore.similarity_search("what is langchain?", k=1)

In [11]:
response = llm.invoke(f"What is Langchain?  You can answer using the following context: {context}")
print(response.content)

Langchain is an open-source project launched in October 2022 by Harrison Chase, who was working at Robust Intelligence at the time. It has since incorporated and raised significant funding, including a $10 million seed investment from Benchmark and over $20 million from Sequoia Capital in April 2023, valuing the startup at at least $200 million.
